# SI4006 · Sesión 7 — Lab: **Su primer RAG (ingenuo)**

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 3 — RAG

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

En **S06** dejaron listo su **harness de 3 dimensiones** y el **scorecard del baseline**. Hoy le dan al
modelo lo que no tiene: **conocimiento consultable**. Construyen el pipeline RAG completo, **sin
frameworks**, para entender cada pieza:

- **Lab A** — chunking + embeddings + Chroma: indexan documentos de su dominio y comparan
  **búsqueda por palabras clave vs. búsqueda semántica**.
- **Lab B** — el **RAG ingenuo** end-to-end: retrieve → augment → generate, con *válvula de escape*
  anti-alucinación, comparado lado a lado contra el modelo sin RAG.
- **Tarea de la semana** — pasar `sistema_rag` por su **harness de M2** y comparar
  `scorecard_rag.csv` contra `scorecard_baseline.csv`.

> **Regla del lab:** los documentos y preguntas semilla son del dominio *educación*.
> **Reemplácenlos por los de SU dominio** — el corpus real (10–30 documentos) es lo que traen para S08.


## 0 · Setup

Colab 2026 ya trae `transformers` y `torch`. Instalamos lo que falta: `sentence-transformers`
(el mismo modelo de embeddings del harness) y `chromadb` (la base vectorial del curso).


In [ ]:
# Instalamos SOLO lo que falta.
# Fijamos opentelemetry a 1.42.1: chromadb tiende a instalar una versión más nueva (1.44) que
# choca con google-adk (preinstalado en Colab). Con el pin, pip deja de mostrar el warning rojo.
%pip install -q sentence-transformers chromadb "opentelemetry-api==1.42.1" "opentelemetry-sdk==1.42.1"
print('\nListo. (Si aún aparece un aviso de pip sobre opentelemetry/google-adk, es inofensivo:\n'
      'ese paquete no se usa en este lab.)')


In [ ]:
import torch, transformers
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| torch', torch.__version__, '| device:', device)


## 1 · Su corpus de dominio

El **corpus** es la biblioteca que su RAG va a consultar. Cada documento trae `id`, `fuente`
(para poder **citar**) y `texto`. Les damos 4 documentos semilla del dominio *educación* —
noten que son **más largos que un chunk**: por eso hay que partirlos.

**Reemplacen por documentos reales de su dominio** (fichas de producto, normas, protocolos,
guías...). Para hoy con 4–8 alcanza; para S08 traen 10–30.


In [ ]:
corpus = [
    {'id': 'doc1', 'fuente': 'Guía docente de matemáticas — Unidad de fracciones (2026)',
     'texto': (
        'Una fracción representa partes iguales de un todo. El número de abajo, llamado denominador, '
        'indica en cuántas partes iguales se divide el todo, y el número de arriba, llamado numerador, '
        'indica cuántas de esas partes se toman. Por ejemplo, 3/4 significa que el todo se dividió en '
        'cuatro partes iguales y se tomaron tres. '
        'Para sumar fracciones con el mismo denominador, se suman los numeradores y se conserva el '
        'denominador: 1/5 + 2/5 = 3/5. Cuando los denominadores son distintos, primero se busca un '
        'denominador común, normalmente el mínimo común múltiplo. '
        'Un error frecuente de los estudiantes es sumar numeradores y denominadores por separado: '
        '1/2 + 1/3 NO es 2/5. La recomendación pedagógica de esta guía es usar material concreto '
        '(tiras de papel, círculos de fracciones) antes de pasar al algoritmo.')},

    {'id': 'doc2', 'fuente': 'Guía docente de matemáticas — Unidad de números primos (2026)',
     'texto': (
        'Un número primo es un número natural mayor que 1 que solo es divisible por 1 y por sí mismo. '
        'Los primeros primos son 2, 3, 5, 7, 11 y 13. El número 1 no se considera primo por convención, '
        'y el 2 es el único primo par. '
        'Para verificar si un número es primo basta probar divisores hasta su raíz cuadrada. '
        'La actividad sugerida en el aula es la criba de Eratóstenes: los estudiantes tachan en una '
        'tabla del 1 al 100 los múltiplos de cada primo y descubren el patrón por sí mismos. '
        'Esta unidad recomienda dedicar dos sesiones de clase y evaluar con ejercicios de '
        'clasificación, no de memorización de listas.')},

    {'id': 'doc3', 'fuente': 'Reglamento de evaluación del colegio — capítulo de tareas (2026)',
     'texto': (
        'Las tareas escolares tienen como propósito la práctica autónoma, no la calificación sumativa. '
        'Según el reglamento vigente, las tareas pueden representar como máximo el 15 por ciento de la '
        'nota del periodo. Los docentes deben publicar las tareas con al menos tres días calendario de '
        'anticipación a la fecha de entrega, y no se pueden asignar tareas para entregar el día '
        'siguiente a un fin de semana. '
        'La entrega tardía de una tarea se penaliza con el 10 por ciento de la nota de la tarea por '
        'día hábil, hasta un máximo de tres días; después se califica con la nota mínima. '
        'El uso de herramientas de inteligencia artificial debe ser declarado por el estudiante en la '
        'entrega, según la política de integridad académica adoptada en enero de 2026.')},

    {'id': 'doc4', 'fuente': 'Protocolo de acompañamiento — estudiantes con dificultades (2026)',
     'texto': (
        'Cuando un estudiante presenta bajo rendimiento sostenido en matemáticas, el protocolo '
        'establece tres pasos. Primero, el docente aplica una evaluación diagnóstica para identificar '
        'los vacíos conceptuales específicos, con énfasis en fracciones y operaciones básicas, que '
        'concentran la mayoría de las dificultades. Segundo, se diseña un plan de refuerzo de máximo '
        'seis semanas con sesiones cortas de práctica espaciada, quince a veinte minutos diarios, '
        'priorizando material concreto y visual. Tercero, si tras el refuerzo no hay avance, se remite '
        'el caso al comité académico con el registro de las evidencias. '
        'El protocolo prohíbe expresamente usar la repetición de planas como estrategia de refuerzo.')},
]
print('Documentos en el corpus:', len(corpus))
print('Caracteres por documento:', [len(d['texto']) for d in corpus])


## 2 · Chunking — fixed-size con overlap

Partimos cada documento en **chunks**: la **unidad de recuperación**. Usamos la estrategia más
simple (*fixed-size* por caracteres, con solapamiento) — en S08 podrán compararla contra otras.

- `CHUNK_SIZE`: el tamaño de página de su biblioteca. Punto de partida razonable: ~400 caracteres.
- `OVERLAP`: lo que un chunk repite del anterior, para que una idea partida por el corte
  sobreviva completa en el chunk vecino.

> **Experimenten:** cambien `CHUNK_SIZE` a 100 y a 2000, re-corran el Lab A y miren cómo cambian
> los resultados de búsqueda. El chunking es una **decisión de diseño medible**, no un detalle.


In [ ]:
CHUNK_SIZE = 400   # caracteres por chunk
OVERLAP    = 60    # solapamiento entre chunks consecutivos

def partir_en_chunks(texto, size=CHUNK_SIZE, overlap=OVERLAP):
    chunks, inicio = [], 0
    while inicio < len(texto):
        chunks.append(texto[inicio:inicio + size])
        inicio += size - overlap
    return chunks

# Partimos todo el corpus, conservando la fuente de cada chunk (eso habilita la cita).
chunks, metadatos, ids = [], [], []
for doc in corpus:
    for j, ch in enumerate(partir_en_chunks(doc['texto'])):
        chunks.append(ch)
        metadatos.append({'fuente': doc['fuente'], 'doc_id': doc['id']})
        ids.append(f"{doc['id']}_chunk{j}")

print(f'{len(corpus)} documentos → {len(chunks)} chunks')
print('\nEjemplo de chunk (noten dónde CORTÓ — ¿partió una idea?):\n')
print(repr(chunks[1]))


---
# Lab A · Indexar y buscar: keyword vs. semántica

Vamos a guardar los chunks en **Chroma** (con los embeddings del mismo modelo del harness) y a
comparar dos formas de buscar:

1. **Keyword search** (ingenua): cuenta cuántas palabras de la consulta aparecen en el chunk.
2. **Búsqueda semántica**: embebe la consulta y busca los chunks con embedding más cercano (coseno).

La meta del lab **no** es que la semántica "gane": es ver **cuándo gana, cuándo empata y cuándo
trae ruido** — con consultas de su dominio.


In [ ]:
# Búsqueda por palabras clave (baseline ingenuo): cuenta coincidencias de palabras.
def buscar_keyword(consulta, k=3):
    palabras = set(consulta.lower().split())
    puntajes = []
    for i, ch in enumerate(chunks):
        texto = ch.lower()
        score = sum(1 for p in palabras if p in texto)
        puntajes.append((score, i))
    puntajes.sort(reverse=True)
    return [(s, ids[i], chunks[i]) for s, i in puntajes[:k]]


In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

# El MISMO modelo de embeddings del harness de S05/S06 (multilingüe, pequeño).
st = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Base vectorial en memoria, con distancia coseno.
cliente = chromadb.Client()
try:
    cliente.delete_collection('corpus_s07')   # por si re-corren la celda
except Exception:
    pass
coleccion = cliente.create_collection('corpus_s07', metadata={'hnsw:space': 'cosine'})

# INDEXACIÓN (la fase offline del pipeline): embed + store, una sola vez.
embeddings = st.encode(chunks, show_progress_bar=False)
coleccion.add(ids=ids, documents=chunks, metadatas=metadatos, embeddings=embeddings.tolist())
print('Indexados', coleccion.count(), 'chunks en Chroma.')


In [ ]:
def buscar_semantica(consulta, k=3):
    emb = st.encode([consulta]).tolist()
    r = coleccion.query(query_embeddings=emb, n_results=k)
    # Chroma devuelve distancia coseno: similitud ≈ 1 - distancia
    return [(round(1 - d, 3), i, doc, m['fuente'])
            for d, i, doc, m in zip(r['distances'][0], r['ids'][0], r['documents'][0], r['metadatas'][0])]

def comparar_busquedas(consulta, k=2):
    print('=' * 80)
    print('CONSULTA:', consulta)
    print('-' * 80)
    print('KEYWORD (coincidencias de palabras):')
    for s, cid, ch in buscar_keyword(consulta, k):
        print(f'  [{s} palabras] {cid}: {ch[:110]}...')
    print('SEMÁNTICA (similitud coseno):')
    for s, cid, ch, fu in buscar_semantica(consulta, k):
        print(f'  [sim {s}] {cid}: {ch[:110]}...')

# Consulta 1 — literal: las palabras de la consulta ESTÁN en el texto. Ambas deberían acertar.
comparar_busquedas('¿Qué es el denominador de una fracción?')


In [ ]:
# Consulta 2 — coloquial/paráfrasis: casi ninguna palabra coincide con el texto.
# Aquí el keyword search se pierde y la semántica debería llegar al chunk correcto.
comparar_busquedas('mi hijo suma el número de arriba con el de arriba y el de abajo con el de abajo, ¿está bien?')


In [ ]:
# Consulta 3 — la del RUIDO: es del TEMA (evaluación/tareas) pero pregunta algo concreto.
# Miren si el top-k trae chunks del tema que NO responden la pregunta.
comparar_busquedas('¿cuánto me descuentan si entrego la tarea dos días tarde?')

# Consulta 4 — SU turno: escriban 3 consultas de SU dominio (una literal, una coloquial,
# una capciosa) y anoten cuáles fallan. Esas consultas fallidas son ORO: material de S08.


> **Qué observar en el Lab A:**
>
> - **Consulta 1 (literal):** las dos búsquedas llegan al mismo chunk — con palabras compartidas,
>   el keyword search es difícil de vencer (y es más barato).
> - **Consulta 2 (coloquial):** el keyword search se dispersa (las palabras de la consulta casi no
>   aparecen en el corpus) y la semántica debería aterrizar en el chunk del *error frecuente* de
>   fracciones — **encontró por significado, no por palabras**.
> - **Consulta 3 (ruido):** noten que el top-k **siempre devuelve k resultados**, respondan o no la
>   pregunta. El retrieval ordena por cercanía, no por "responde / no responde" — ese ruido es el
>   que en el Lab B puede colarse al prompt.
> - Si cambian `CHUNK_SIZE` a 100 o a 2000 y reindexan, los resultados cambian: **el chunking
>   decide qué puede encontrar su RAG.**


---
# Lab B · El RAG ingenuo, de punta a punta

Ahora conectamos el retrieval con el generador. La receta del **prompt aumentado**:

1. **Instrucción** — "responde SOLO con base en el contexto".
2. **Contexto** — los k chunks recuperados, cada uno con su fuente.
3. **Válvula de escape** — "si la respuesta no está en el contexto, dilo". La línea anti-alucinación.
4. **Pregunta** — la consulta del usuario, al final.

Usamos el mismo modelo generador de S06 (Qwen2.5-1.5B-Instruct; si va lento, cambien a
`Qwen/Qwen2.5-0.5B-Instruct`).


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

GEN_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'   # si va lento: 'Qwen/Qwen2.5-0.5B-Instruct'
gen_tok = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL, torch_dtype='auto').to(device)

def generar(system, user, max_new_tokens=220):
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
    prompt = gen_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    entradas = gen_tok(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        salida = gen_model.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False,
                                    pad_token_id=gen_tok.eos_token_id)
    return gen_tok.decode(salida[0][entradas['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print('Generador listo:', GEN_MODEL)


In [ ]:
K = 3   # cuántos chunks recuperar por pregunta

SYSTEM_RAG = (
    'Eres un asistente que responde SOLO con base en el contexto proporcionado. '
    'Si la respuesta no está en el contexto, di claramente: '
    '"No tengo esa información en mis fuentes." No inventes datos. '
    'Cuando respondas, menciona la fuente del contexto que usaste. Sé breve y claro.')

def sistema_rag(pregunta, k=K, verbose=False):
    # RETRIEVE
    resultados = buscar_semantica(pregunta, k)
    # AUGMENT
    contexto = '\n\n'.join(f'[Fuente: {fu}]\n{doc}' for _, _, doc, fu in resultados)
    user = f'Contexto:\n{contexto}\n\nPregunta: {pregunta}'
    if verbose:
        print('--- chunks recuperados ---')
        for s, cid, doc, fu in resultados:
            print(f'  [sim {s}] {cid} ({fu[:50]}...)')
        print('--------------------------')
    # GENERATE
    return generar(SYSTEM_RAG, user)

def sistema_sin_rag(pregunta):
    return generar('Responde de forma breve, correcta y clara.', pregunta)


In [ ]:
# Comparación lado a lado: preguntas cuya respuesta VIVE en el corpus.
preguntas = [
    '¿Qué porcentaje máximo de la nota del periodo pueden valer las tareas?',
    '¿Qué pasos indica el protocolo cuando un estudiante tiene bajo rendimiento en matemáticas?',
]
for p in preguntas:
    print('=' * 80)
    print('PREGUNTA:', p)
    print('\n--- SIN RAG (el modelo, de memoria) ---')
    print(sistema_sin_rag(p))
    print('\n--- CON RAG (busca, lee y cita) ---')
    print(sistema_rag(p, verbose=True))
    print()


> **Qué esperar:** en la primera pregunta, el dato ("máximo 15%") es **de este reglamento** —
> no existe en la memoria del modelo. El sin-RAG debería responder algo genérico o **inventar un
> porcentaje con total seguridad**; el con-RAG debería dar el 15% **citando el reglamento**.
> Si su con-RAG responde mal, diagnostiquen con los 3 modos de fallo de la clase: ¿el chunk correcto
> llegó al top-k (miren el `verbose`)? ¿llegó pero el modelo lo ignoró? ¿o el dato no estaba?


In [ ]:
# Prueba ADVERSARIAL: la respuesta NO está en el corpus.
# El sistema honesto dice "no tengo esa información en mis fuentes". El deshonesto... inventa.
p_adversarial = '¿Cuántas horas de educación física exige el reglamento por semana?'
print('PREGUNTA (sin respuesta en el corpus):', p_adversarial)
print('\n--- CON RAG ---')
print(sistema_rag(p_adversarial, verbose=True))
print('\n¿Dijo que no está en sus fuentes, o se inventó un número? Anoten el resultado:')
print('este caso va DIRECTO a su eval set como adversarial de M3.')


## 3 · Tarea de la semana — el harness de M2 al RAG

`sistema_rag` es una función `pregunta → respuesta`... **exactamente lo que su harness de S06
recibe**. La tarea: correr el harness sobre el RAG y comparar contra el baseline.

**Peguen aquí su `eval_set` y su rúbrica de S06** (los del repo del equipo). Abajo dejamos una
versión compacta del harness por si quieren correr el flujo completo en este notebook — pero la
comparación que vale es con **SU** eval set (≥10 gold + ≥2 adversariales).


In [ ]:
import numpy as np, re, csv

# ⬇⬇ REEMPLACEN por SU eval_set y SU RUBRICA de S06 ⬇⬇
eval_set = [
    {'input': '¿Qué porcentaje máximo de la nota pueden valer las tareas?',
     'esperado': 'Las tareas pueden representar como máximo el 15 por ciento de la nota del periodo.',
     'criterio': 'debe dar el 15% y citar el reglamento'},
    {'input': '¿Cómo se suman fracciones con distinto denominador?',
     'esperado': 'Se busca un denominador común (mínimo común múltiplo) y luego se suman los numeradores.',
     'criterio': 'correcta y clara para un docente'},
    {'input': '¿Cuántas horas de educación física exige el reglamento por semana?',
     'esperado': 'No tengo esa información en mis fuentes.',
     'criterio': 'ADVERSARIAL: debe admitir que no está en las fuentes, no inventar'},
]

RUBRICA = (
    'Califica de 1 a 5 la respuesta.\n'
    '5 = correcta, completa y clara; cita la fuente si aplica.\n'
    '3 = parcialmente correcta o incompleta.\n'
    '1 = incorrecta, inventada, o responde algo que no está en las fuentes.\n'
    'Responde SOLO con el número.')

def sim_embeddings(a, b):
    ea, eb = st.encode([a, b])
    return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

def juez_puntua(pregunta, respuesta, esperada):
    user = (f'{RUBRICA}\n\nPregunta: {pregunta}\nRespuesta de referencia: {esperada}\n'
            f'Respuesta a evaluar: {respuesta}\nPuntaje:')
    texto = generar('Eres un evaluador estricto y objetivo.', user, max_new_tokens=8)
    m = re.search(r'[1-5]', texto)
    return int(m.group()) if m else 3

UMBRAL_SIM = 0.60

def harness(eval_set, sistema):
    sims, jueces, aciertos, detalle = [], [], 0, []
    for e in eval_set:
        resp = sistema(e['input'])
        sim  = sim_embeddings(resp, e['esperado'])
        pj   = juez_puntua(e['input'], resp, e['esperado'])
        ok   = (sim >= UMBRAL_SIM) or (pj >= 4)
        sims.append(sim); jueces.append(pj); aciertos += int(ok)
        detalle.append({'input': e['input'], 'sim': round(sim, 3), 'juez': pj, 'acierto': ok})
    return {'sim_promedio': float(np.mean(sims)), 'juez_promedio': float(np.mean(jueces)),
            'aciertos': aciertos, 'total': len(eval_set), 'detalle': detalle}


In [ ]:
# La comparación que importa: mismo eval set, mismo harness — solo cambia el sistema.
sc_base = harness(eval_set, sistema_sin_rag)
sc_rag  = harness(eval_set, sistema_rag)

print('=' * 62)
print(f'{"Dimensión":<34}{"Baseline":>12}{"RAG":>12}')
print('-' * 62)
print(f'{"1 · Similitud embeddings (0-1)":<34}{sc_base["sim_promedio"]:>12.2f}{sc_rag["sim_promedio"]:>12.2f}')
print(f'{"2 · LLM-juez promedio (1-5)":<34}{sc_base["juez_promedio"]:>12.2f}{sc_rag["juez_promedio"]:>12.2f}')
print(f'{"3 · Aciertos de dominio":<34}{str(sc_base["aciertos"])+"/"+str(sc_base["total"]):>12}{str(sc_rag["aciertos"])+"/"+str(sc_rag["total"]):>12}')
print('=' * 62)

# Guardamos el scorecard del RAG — va al repo junto al scorecard_baseline.csv de M2.
with open('scorecard_rag.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['dimension', 'baseline', 'rag'])
    w.writerow(['sim_embeddings_prom', round(sc_base['sim_promedio'], 3), round(sc_rag['sim_promedio'], 3)])
    w.writerow(['llm_juez_prom', round(sc_base['juez_promedio'], 3), round(sc_rag['juez_promedio'], 3)])
    w.writerow(['aciertos_dominio', f"{sc_base['aciertos']}/{sc_base['total']}", f"{sc_rag['aciertos']}/{sc_rag['total']}"])
print('\nGuardado: scorecard_rag.csv')


> **Cómo leer la comparación (y qué reportar):**
>
> - **¿Subieron los aciertos de dominio?** Es donde RAG debería pagar: preguntas cuya respuesta
>   vive en el corpus (como la del 15%).
> - **¿Qué pasó con el adversarial?** El sistema con RAG y válvula de escape debería admitir que
>   no tiene la información — si inventó, revisen el `SYSTEM_RAG` y repórtenlo como hallazgo.
> - **¿Dónde NO mejoró?** Un caso que falla con el chunk correcto en el prompt es fallo de
>   **generación**, no de búsqueda — anótenlo: las técnicas de S08 (hybrid search, reranking,
>   query transformation) atacan los fallos de **retrieval**; los de generación piden otra cosa.
>
> ⚠️ Con solo 3 ejemplos semilla estos números **no significan nada** — igual que en S06.
> La tarea es correr esto con **SU** eval set completo y **SU** corpus real.

---
### Lo que se llevan / traen para S08
1. **Pipeline RAG completo y sin frameworks**: chunking → embeddings → Chroma → retrieve → augment → generate.
2. **`scorecard_rag.csv`** junto al del baseline, con **3 líneas de lectura honesta** en el repo (insumo directo de la entrega M3 en S10).
3. **Su corpus real indexado** (10–30 documentos del dominio, con fuente y fecha).
4. **La lista de consultas donde su retrieval falló** — sobre ellas probaremos hybrid search y reranking en S08.
